# POLITSKY24 Dataset Accessing and Testing

In [4]:
import os
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Change path on .env file on your local machine
data_path = Path(os.getenv("DATA_PATH"))

In [5]:
# Access the parquet files in the data path

parquet_files = list(data_path.glob("*.parquet"))
parquet_files

[WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_full_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_network_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_history_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_list_for_stance_detection.parquet')]

In [6]:
import polars as pl

# Load each dataset manually into individual variables
user_network_df = pl.read_parquet(data_path / "user_network_dataset.parquet")
user_post_history_df = pl.read_parquet(data_path / "user_post_history_dataset.parquet")
user_post_list_df = pl.read_parquet(data_path / "user_post_list_for_stance_detection.parquet")

# Stance label datasets
llm_full_stance_df = pl.read_parquet(data_path / "llm_annotated_full_user_stance_dataset.parquet")
llm_val_stance_df = pl.read_parquet(data_path / "llm_annotated_validation_user_stance_dataset.parquet")
human_val_stance_df = pl.read_parquet(data_path / "human_annotated_validation_user_stance_dataset.parquet")

# Stance relevancy datasets (Note: surround filename strings in quotes)
human_val_relevancy_post_df = pl.read_parquet(
    data_path / "human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet"
)
human_val_relevancy_query_df = pl.read_parquet(
    data_path / "human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet"
)

In [7]:
user_network_df.schema

Schema([('SourceUserId', Int64),
        ('TargetUserId', Int64),
        ('InteractionType', String),
        ('Count', Int64)])

In [8]:
user_post_history_df.schema

Schema([('PostId', Int64),
        ('UserId', Int64),
        ('PostTime', String),
        ('Content', String),
        ('Hashtags', List(String)),
        ('Languages', List(String)),
        ('Mentions', List(Int64)),
        ('IsRepost', Boolean),
        ('SourceUserId', Int64),
        ('IsQuote', Boolean),
        ('QuoteId', Int64),
        ('QuoteUserId', Int64),
        ('QuoteContent', String),
        ('IsReply', Boolean),
        ('ParentId', Int64),
        ('ParentUserId', Int64)])

In [24]:
user_post_history_df.select(pl.all().n_unique()).to_dicts()[0]

{'PostId': 14830231,
 'UserId': 8561,
 'PostTime': 18230517,
 'Content': 13138778,
 'Hashtags': 285770,
 'Languages': 1193,
 'Mentions': 88178,
 'IsRepost': 2,
 'SourceUserId': 217858,
 'IsQuote': 2,
 'QuoteId': 975337,
 'QuoteUserId': 105488,
 'QuoteContent': 910916,
 'IsReply': 2,
 'ParentId': 5472655,
 'ParentUserId': 291653}

In [26]:
# Exact duplicate rows (list columns may need to be dropped/converted first)
user_post_history_df.is_duplicated().sum()

# Rows sharing a PostId
dupes = user_post_history_df.filter(pl.col("PostId").is_duplicated())
dupes.sort("PostId").head(20)

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
18,16171,"""2023-12-03T01:11:35.122Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
18,71390,"""2023-12-04T22:26:06.295Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
47,9,"""2023-06-22T00:48:37.165Z""","""I used a Chrome/Edge extension…",[],[],[],false,9,false,-1,-1,"""""",false,15691770,343645
47,285903,"""2023-06-22T00:49:34.781Z""","""I used a Chrome/Edge extension…",[],[],[],true,9,false,-1,-1,"""""",false,15691770,343645
279,9,"""2023-07-06T17:04:05.906Z""","""Should be required that PD off…",[],"[""en""]",[],false,9,false,-1,-1,"""""",false,4244240,90714
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999,141415,"""2023-10-03T01:52:13.292Z""","""Let's talk about how domestic …",[],"[""en""]",[],true,9,false,-1,-1,"""""",false,-1,-1
1000,90924,"""2023-10-03T01:47:26.282Z""","""Going along with Sheryl's post…",[],"[""en""]",[],true,9,false,-1,-1,"""""",true,999,9
1000,9,"""2023-10-03T00:58:13.600Z""","""Going along with Sheryl's post…",[],"[""en""]",[],false,9,false,-1,-1,"""""",true,999,9


In [ ]:


user_po

In [25]:
user_post_history_df.shape

(18416787, 16)

In [9]:
user_post_list_df.schema

Schema([('UserId', Int64), ('PostIdList', List(Int64))])

In [10]:
llm_full_stance_df.schema

Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [11]:
llm_val_stance_df.schema


Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [12]:
human_val_stance_df.schema

Schema([('UserId', Int64), ('Trump', String), ('Harris', String)])

In [13]:
human_val_relevancy_post_df.schema

Schema([('PostId', Int64), ('Trump', Int64), ('Harris', Int64)])

In [14]:
human_val_relevancy_query_df.schema

Schema([('Query', String),
        ('PostId', Int64),
        ('StanceRelevanceLabel', String)])

## DuckDB Testing

In [15]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [16]:
import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Redirect DuckDB's temp/spill files to your D drive instead of C:
# con.execute("SET temp_directory = 'D:/Thesis-related/POLITSKY24/duckdb_temp.tmp';")


query = """
SELECT *
FROM user_post_history_df as uph
LEFT JOIN llm_full_stance_df as lfs
    ON uph.UserId = lfs.UserId
LIMIT 10
"""

joined_df = con.execute(query).fetchdf()


In [17]:
print(user_post_history_df.shape)
print(llm_full_stance_df.shape)
print(joined_df.shape)

(18416787, 16)
(16044, 10)
(10, 26)


In [ ]:
joined_df.head(10)

,PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,...,UserId_1,TargetEntity,Context,LLMAnswerContent,LLMAnswerUsage,SourcePosts,Spans,Reason,StanceLabel,ConfidenceLevel
0,7840946,167406,2024-11-19T23:12:12.092Z,it’s a tragedy of the commons. so. anyone and ...,[],[en],[],False,167406,False,...,167406,Trump,[@anonymous Trump is no different from HW in t...,"{\n ""stance"": ""Neither"",\n ""reason"": ""Th...","{'input_tokens': 900, 'output_tokens': 44, 'to...",<NA>,<NA>,The tweets do not provide a clear stance towar...,Neither,NaN
1,17079336,377198,2024-11-19T23:12:11Z,"Notícia da @anonymous\n\n""Xi Jinping arrives i...",[],[],[],False,377198,False,...,377198,Trump,"[Notícia da @anonymous\n\n""Trump nega disputar...","{\n ""stance"": ""Neither"",\n ""reason"": ""Th...","{'input_tokens': 904, 'output_tokens': 51, 'to...",<NA>,<NA>,The provided tweets do not contain any direct ...,Neither,NaN
2,9179962,196837,2024-11-19T23:12:11.298Z,Have you run it by the Godless Democratic Part...,[],[en],[],False,196837,False,...,196837,Trump,[Unpopular opinion: Trump will not be a dicta...,"{\n ""source_tweets"": [""Trump will fix Ameri...","{'input_tokens': 834, 'output_tokens': 182, 't...",[Trump will fix America and get rid of the fai...,[Trump will fix America and get rid of the fai...,The user expresses strong support for Donald T...,Favor,0.95
3,15477859,337604,2024-11-19T23:12:10.942Z,They are hell bent on it. And what's with the ...,[],[en],[],False,337604,False,...,337604,Trump,"[I appreciate my TRUMP vote more than ever, se...","{\n ""source_tweets"": [""I appreciate my TRUM...","{'input_tokens': 815, 'output_tokens': 165, 't...","[I appreciate my TRUMP vote more than ever, se...","[I appreciate my TRUMP vote more than ever, Wh...",The user expresses strong support for Donald T...,Favor,0.95
4,9528000,207668,2024-11-19T23:12:10.499Z,Why did you stalk all the way over here to do ...,[],[en],[],False,207668,False,...,207668,Trump,"[If he violates the constitution, he's no bett...","{\n ""source_tweets"": [""If he violates the c...","{'input_tokens': 911, 'output_tokens': 250, 't...","[If he violates the constitution, he's no bett...","[no better than trump, zero sympathy for anyon...",The user expresses negative sentiments towards...,Against,0.95
5,1684791,84980,2024-11-19T23:58:06.973Z,So good and funny!,[],[en],[],True,36231,False,...,84980,Trump,"[FUCK TRUMP!\n#Recount2024\n🦋💙🦋, Smash em if y...","{\n ""source_tweets"": [""FUCK TRUMP!"", ""Seems...","{'input_tokens': 800, 'output_tokens': 176, 't...","[FUCK TRUMP!, Seems like sexual assault is a p...","[FUCK TRUMP!, sexual assault is a pre-requisit...",The user expresses strong negative sentiments ...,Against,0.95
6,6601773,350038,2024-11-19T23:16:30.635Z,Did I offend you in some with my simple reques...,[],[en],[],True,139233,False,...,350038,Trump,[I will never forgive anyone who voted for tru...,"{\n ""source_tweets"": [""I will never forgive...","{'input_tokens': 770, 'output_tokens': 138, 't...",[I will never forgive anyone who voted for tru...,"[never forgive anyone who voted for trump, #fu...",The user expresses strong negative sentiments ...,Against,0.95
7,17618975,388277,2024-11-19T23:12:07.392Z,Holy shit that's creepy,[],[en],[],False,388277,False,...,388277,Trump,"[I have never liked Donald Trump, but unless D...","{\n ""source_tweets"": [""I have never liked D...","{'input_tokens': 813, 'output_tokens': 160, 't...","[I have never liked Donald Trump, but unless D...","[I have never liked Donald Trump, Donald Trump...",The user expresses a dislike for Donald Trump ...,Against,0.90
8,4214196,21649,2024-11-19T23:26:29.875Z,"More ""good news"" for writers... 🙃",[],[en],[],True,89798,True,...,21649,Trump,[Magats are a wee bit mad that I’m posting the...,"{\n ""source_tweets"": [""Magats are a wee bit...","{'input_tokens': 760, 'output_tokens': 240, 't...",[Magats are a wee bit mad that I’m posting the...,[Magats are a wee bit mad that I’